<h3>Importing Libraries</h3>

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack
from pathlib import Path

<h3>Loading Data</h3>

In [14]:
PROJECT_ROOT = Path.cwd()
TRAIN_CSV = PROJECT_ROOT / "train.csv"
TEST_CSV = PROJECT_ROOT / "test.csv"
SAMPLE_SUBMISSION_CSV = PROJECT_ROOT / "sample_submission.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)

train_df.head(5)

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


<h3>Data Analysis & Exploration</h3>

In [15]:
# 57477 Rows & 9 Features
train_df.shape

(57477, 9)

In [16]:
train_df.columns
# id = Unique ID. Doesn't start with 0-1 and gradually increase.
# model_a = First model for comparison.
# model_b = Second model for comparison.
# prompt = Entered prompt for getting an answer
# response_a = Output created by model_a
# response_b = Output created by model_b
# winner_model_a = One Hot Encoded. Shows if model_a won.
# winner_model_b = One Hot Encoded. Shows if model_b won.
# winner_tie = One Hot Encoded. Shows if there was a tie between models.

Index(['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b',
       'winner_model_a', 'winner_model_b', 'winner_tie'],
      dtype='object')

In [17]:
# Checking if the data is balanced
print((train_df["winner_model_a"] == 1).sum())
print((train_df["winner_model_b"] == 1).sum())
print((train_df["winner_tie"] == 1).sum())

20064
19652
17761


<h3>Checking for Duplicates, Missing, Dirty Data</h3>

In [18]:
train_df.duplicated(subset=["prompt"]).sum()
# 5743 Rows missing, optional drop below:
# train_df.drop_duplicates(subset=["prompt"])

np.int64(5743)

In [19]:
# Checking for null values
train_df.isnull().sum()

id                0
model_a           0
model_b           0
prompt            0
response_a        0
response_b        0
winner_model_a    0
winner_model_b    0
winner_tie        0
dtype: int64

In [20]:
# Checking for empty prompts
(train_df['prompt'].str.strip() == "").sum()

np.int64(0)

<h3>Training</h3>

In [21]:
def format_input(row):
    return f"""
### Prompt:
{row["prompt"]}

### Response A
{row["response_a"]}

### Response B
{row["response_b"]}
"""

train_df["text"] = train_df.apply(format_input, axis=1)

In [22]:
train_df["label"] = np.argmax(
    train_df[["winner_model_a", "winner_model_b", "winner_tie"]].values,
    axis=1
)

train_df["label"].value_counts()

label
0    20064
1    19652
2    17761
Name: count, dtype: int64

In [23]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [24]:
train_df["fold"] = -1

for fold, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df["label"])
):
    train_df.loc[val_idx, "fold"] = fold

In [25]:
train_df["fold"].value_counts()

fold
0    11496
1    11496
4    11495
3    11495
2    11495
Name: count, dtype: int64

In [26]:
def swap_rows(df):
    swapped = df.copy()

    swapped["response_a"], swapped["response_b"] = df["response_b"], df["response_a"]

    swapped["winner_model_a"], swapped["winner_model_b"] = (
        df["winner_model_b"],
        df["winner_model_a"]
    )

    swapped["label"] = np.argmax(
        swapped[["winner_model_a", "winner_model_b", "winner_tie"]].values,
        axis=1
    )

    return swapped

train_df = pd.concat([train_df, swap_rows(train_df)], ignore_index=True)

In [29]:
preds = np.zeros((len(test_df), 3))

for fold in range(5):
    print(f"Inference Fold {fold}")

    train_data = train_df[train_df["fold"] != fold]

    X_train = train_data["text"]
    y_train = train_data["label"]

    vectorizer = TfidfVectorizer(
        max_features=100000,
        ngram_range=(1, 2),
        analyzer="word",
        sublinear_tf=True
    )

    char_vectorizer = TfidfVectorizer(
        max_features=20000,
        analyzer="char",
        ngram_range=(3, 5)
    )

    X_train_word = vectorizer.fit_transform(X_train)
    X_train_char = char_vectorizer.fit_transform(X_train)

    X_train_vec = hstack([X_train_word, X_train_char])

    model = LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1)
    model.fit(X_train_vec, y_train)

    X_test = test_df.apply(format_input, axis=1)

    X_test_word = vectorizer.transform(X_test)
    X_test_char = char_vectorizer.transform(X_test)

    X_test_vec = hstack([X_test_word, X_test_char])

    preds += model.predict_proba(X_test_vec)

preds /= 5

Inference Fold 0
Inference Fold 1
Inference Fold 2
Inference Fold 3
Inference Fold 4


<h3>Evaluation and Output</h3>

In [30]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": preds[:, 0],
    "winner_model_b": preds[:, 1],
    "winner_tie": preds[:, 2],
})

# Save and preview the submission
submission.to_csv("submission.csv", index=False)
submission.head()

,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.267403,0.267409,0.465188
1,211333,0.351384,0.351368,0.297248
2,1233961,0.434420,0.434442,0.131138
